# Python Code Error Debugger with QLoRA Fine-tuning

Fine-tune Meta's Llama 3.2 3B model to debug Python code errors using Stack Overflow data.

**Project Features:**
- Dataset from Stack Overflow Python Q&A
- QLoRA fine-tuning
- Code error detection and correction
- Interactive Gradio UI

**Base Model:** `meta-llama/Llama-3.2-3B-Instruct`

**Dataset:** `koutch/stackoverflow_python`

## Part 1. Setup and Installation

In [1]:
!pip install transformers accelerate peft bitsandbytes datasets trl gradio torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 42.8 MB/s eta 0:00:00


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import gradio as gr
from google.colab import userdata
from huggingface_hub import login
import re
from peft import PeftModel

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Part 2. Loading and Preprocessing Dataset

In [4]:
data = load_dataset("koutch/stackoverflow_python", split="train")
print(len(data))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005-3d48b7eced91a0(…):   0%|          | 0.00/121M [00:00<?, ?B/s]

data/train-00001-of-00005-56eee9940d3dd0(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00002-of-00005-2683a3aec26c96(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00003-of-00005-dfcfc12a49338d(…):   0%|          | 0.00/186M [00:00<?, ?B/s]

data/train-00004-of-00005-69e77510bada75(…):   0%|          | 0.00/193M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/987122 [00:00<?, ? examples/s]

987122


In [5]:
def error_filter(sample):
  contents = sample['title'].lower() + " " + sample['question_body'].lower()
  keywords = ['error', 'exception', 'bug', 'traceback', 'fails', 'broken', 'doesnt work', "doesn't work", 'issue', 'problem']
  return any(keyword in contents for keyword in keywords)

filtered_data = data.filter(error_filter)
print(len(filtered_data))

Filter:   0%|          | 0/987122 [00:00<?, ? examples/s]

431829


In [6]:
def format_instruction(sample):
    problem = sample['title']
    code_body = sample['question_body']
    solution = sample['answer_body']

    code_block = re.search(r'```python\n(.*?)\n```', code_body, re.DOTALL)
    code = code_block.group(1) if code_block else code_body[:500]

    instruction = f"""Debug this piece of Python code and explain the error.

Question: {problem}

Code:
```python
{code[:400]}
```

Provide:
1. Explanation of the error
2. Corrected code"""

    response = solution[:600]

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>"""

    return {"text": prompt}

formatted_data = filtered_data.map(format_instruction)
formatted_data = formatted_data.remove_columns([col for col in formatted_data.column_names if col != 'text'])

Map:   0%|          | 0/431829 [00:00<?, ? examples/s]

In [7]:
target_size = 12000
formatted_data = formatted_data.shuffle(seed=42).select(range(target_size))
train_dataset = formatted_data.select(range(int(0.9 * len(formatted_data))))
eval_dataset = formatted_data.select(range(int(0.9 * len(formatted_data)), len(formatted_data)))

print(len(train_dataset))
print(len(eval_dataset))

10800
1200


## Part 3. Loading Base Model with QLoRA Configuration

In [8]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", trust_remote_code=True)

model = prepare_model_for_kbit_training(model)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [23]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_parameters = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters = ({trainable_parameters / all_parameters * 100:.2f}%)")

Trainable parameters = (0.51%)


## Part 4. Training Configuration

In [25]:
training_args = SFTConfig(
    output_dir="./code-debugger-llama",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    bf16=True,
    tf32=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    eval_strategy="epoch",
    max_length=512,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)


## Part 5. Training the Model

In [26]:
trainer.train()
print("Complete")
trainer.save_model("./code-debugger-final")
tokenizer.save_pretrained("./code-debugger-final")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.598400,No log
2,1.534400,No log
3,1.514500,No log


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Complete


('./code-debugger-final/tokenizer_config.json',
 './code-debugger-final/special_tokens_map.json',
 './code-debugger-final/chat_template.jinja',
 './code-debugger-final/tokenizer.json')

In [27]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto")

fine_tuned_model = PeftModel.from_pretrained(base_model, "./code-debugger-final")
fine_tuned_model = fine_tuned_model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.l

## Part 7. Testing and Gradio Interface

In [30]:
def debug_code(code, error_message=""):
    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Debug this Python code and explain the error.

Code:
```python
{code}
```

Error message: {error_message if error_message else "No error message provided"}

Provide:
1. An explanation of the error
2. Corrected python code<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = fine_tuned_model.generate(**inputs, max_new_tokens=400, temperature=0.3, do_sample=True, top_p=0.9)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant")[-1].strip()

    return response

In [31]:
test_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

result = calculate_average([])
print(result)
"""

print(debug_code(test_code, "ZeroDivisionError: division by zero"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Error Explanation**

The error occurs because you're attempting to divide by zero. In the `calculate_average` function, you're trying to calculate the average of a list of numbers, but you're not checking if the input list is empty before attempting to calculate the average. When the input list is empty, `len(numbers)` returns 0, and you're trying to divide by 0, which is mathematically undefined.

**Corrected Code**

Here's the corrected code:
```python
def calculate_average(numbers):
    """
    Calculate the average of a list of numbers.

    Args:
        numbers (list): A list of numbers.

    Returns:
        float: The average of the input list.
    """
    if not numbers:  # Check if the list is empty
        raise ValueError("Cannot calculate average of an empty list")
    total = sum(numbers)  # Use the built-in sum function
    return total / len(numbers)

result = calculate_average([1, 2, 3, 4, 5])
print(result)  # Output: 3.0
```
**Changes Made**

1. Added a check at the

In [35]:
def gradio_debug(code_input, error_input):
    if not code_input.strip():
        return "Please provide code to debug."
    return debug_code(code_input, error_input)

with gr.Blocks() as ui:
    gr.Markdown("# Python Code Debugger")

    code_input = gr.Code(label="Your Python Code", language="python", lines=12)
    error_input = gr.Textbox(label="Error Message (Optional)", placeholder="Paste error traceback here...", lines=3)
    output = gr.Markdown(label="Debug Output")

    debug_button = gr.Button("Debug Code")
    debug_button.click(fn=gradio_debug, inputs=[code_input, error_input], outputs=output)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2fbd2036da5514323c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
